# Texture Anomaly Detection

Isolation Forest learns normal texture and flags unusual real textures.

## Step 1: Import libraries

IsolationForest is unsupervised.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

from sklearn.ensemble import IsolationForest

## Step 2: Load images

Labels are only for evaluation.

In [ ]:
DATASET_DIR=Path("../datasets/08_texture_anomaly_detection")
def load_images(path, size=(128,128)):
    images, labels = [], []
    for class_dir in sorted(path.iterdir()):
        if class_dir.is_dir():
            for fp in sorted(class_dir.glob("*.png")):
                images.append(np.array(Image.open(fp).convert("RGB").resize(size)))
                labels.append(class_dir.name)
    return np.array(images), np.array(labels)

images,labels=load_images(DATASET_DIR)

## Step 3: Extract LBP texture features

LBP summarizes local patterns.

In [ ]:
from skimage.feature import local_binary_pattern
def lbp_feat(im):
    gray=np.mean(im,axis=2).astype(np.uint8)
    lbp=local_binary_pattern(gray,16,2,method="uniform")
    h,_=np.histogram(lbp.ravel(),bins=np.arange(0,19),range=(0,18),density=True)
    return h
features=np.array([lbp_feat(im) for im in images])
print(features.shape)


## Step 4: Train on normal examples only

The model learns normality.

In [ ]:
normal=features[labels=="normal"]
model=IsolationForest(n_estimators=250,contamination=.25,random_state=42).fit(normal)

## Step 5: Evaluate anomaly predictions

1 means normal and -1 means anomaly.

In [ ]:
raw=model.predict(features)
pred=np.where(raw==1,"normal","anomaly")
print("Accuracy:",accuracy_score(labels,pred))
ConfusionMatrixDisplay.from_predictions(labels,pred); plt.show()

# Test One Single Texture for Anomaly Detection

This section predicts whether one separate texture image is normal or anomalous.

Isolation Forest returns `1` for normal and `-1` for anomaly. The same LBP features used for training are applied to the new image.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

single_image_path = Path(
    "../datasets/08_texture_anomaly_detection/anomaly/000_original.png"
)

single_image = Image.open(single_image_path).convert("RGB")
single_image = single_image.resize((128, 128))
single_image_array = np.array(single_image)

single_features = lbp_feat(single_image_array)
single_features_2d = single_features.reshape(1, -1)

raw_prediction = model.predict(single_features_2d)[0]
predicted_label = "normal" if raw_prediction == 1 else "anomaly"
anomaly_score = model.decision_function(single_features_2d)[0]

print("Raw prediction:", raw_prediction)
print("Predicted label:", predicted_label)
print("Decision score:", round(anomaly_score, 4))

plt.figure(figsize=(5, 5))
plt.imshow(single_image_array)
plt.title(f"Prediction: {predicted_label}\nScore: {anomaly_score:.4f}")
plt.axis("off")
plt.show()


## Test one single texture anomaly image 

This section applies the same LBP texture features used during training and predicts whether one image is normal or anomalous. Isolation Forest returns `1` for normal and `-1` for anomaly.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

single_image_path = Path("../datasets/08_texture_anomaly_detection/anomaly/000_original.png")

if not single_image_path.exists():
    raise FileNotFoundError(
        f"Image not found: {single_image_path.resolve()}"
    )

single_image = Image.open(single_image_path).convert("RGB")
single_image = single_image.resize((128, 128))
single_image_array = np.array(single_image)

single_features = lbp_feat(single_image_array)
single_features_2d = np.asarray(single_features).reshape(1, -1)

raw_prediction = model.predict(single_features_2d)[0]
predicted_label = "normal" if raw_prediction == 1 else "anomaly"

print("Raw prediction:", raw_prediction)
print("Predicted label:", predicted_label)

if hasattr(model, "decision_function"):
    anomaly_score = model.decision_function(single_features_2d)[0]
    print("Decision score:", round(float(anomaly_score), 4))
else:
    anomaly_score = None

plt.figure(figsize=(5, 5))
plt.imshow(single_image_array)
title = f"Prediction: {predicted_label}"
if anomaly_score is not None:
    title += f"\nScore: {anomaly_score:.4f}"
plt.title(title)
plt.axis("off")
plt.show()
